# SemanticEmbed: CI/CD & Data Pipeline Analysis

Find structural fragility in build graphs, deployment pipelines, and ETL workflows before they cause cascading failures.

[GitHub](https://github.com/jmurray10/semanticembed-sdk)

In [ ]:
!pip install -q semanticembed
from semanticembed import encode, report, drift
print("Ready.")

---
## Example 1: CI/CD Build Pipeline

A monorepo CI pipeline with shared build steps, test stages, and deployment targets. Where does a failure cascade?

In [ ]:
# CI/CD pipeline as a directed graph
ci_pipeline = [
    # Source triggers
    ("git-push",          "lint"),
    ("git-push",          "typecheck"),
    ("git-push",          "security-scan"),
    
    # Build stage
    ("lint",              "build-frontend"),
    ("lint",              "build-backend"),
    ("typecheck",         "build-frontend"),
    ("typecheck",         "build-backend"),
    
    # Test stage
    ("build-frontend",    "unit-tests-fe"),
    ("build-backend",     "unit-tests-be"),
    ("build-frontend",    "integration-tests"),
    ("build-backend",     "integration-tests"),
    
    # Deploy stage
    ("unit-tests-fe",     "deploy-staging"),
    ("unit-tests-be",     "deploy-staging"),
    ("integration-tests", "deploy-staging"),
    ("security-scan",     "deploy-staging"),
    
    # Production
    ("deploy-staging",    "smoke-tests"),
    ("smoke-tests",       "deploy-prod"),
]

result = encode(ci_pipeline)
print("CI/CD PIPELINE STRUCTURAL ANALYSIS")
print("=" * 50)
print(result.table)
print()
print(report(result))

---
## What to Look For

| Pattern | CI/CD meaning | Risk |
|---------|--------------|------|
| **High criticality** | This step is on most build paths | Flaky test here blocks everything |
| **High fanout** | This step triggers many downstream jobs | A slow build here cascades delays |
| **Low independence** | Only step at this pipeline stage | No parallel fallback if it fails |
| **Convergence sink** | Many steps feed into this one | Bottleneck -- all paths wait here |

In [ ]:
# Which steps are structural bottlenecks?
dim_names = ["depth", "independence", "hierarchy", "throughput", "criticality", "fanout"]

print("CONVERGENCE BOTTLENECKS (everything waits here):")
print("-" * 50)
for node, v in result.vectors.items():
    if v[5] < 0.3 and v[3] > 0.05:  # low fanout + meaningful throughput
        print(f"  {node:<25} throughput={v[3]:.3f}  fanout={v[5]:.3f}")

print()
print("AMPLIFICATION POINTS (trigger many downstream):")
print("-" * 50)
for node, v in result.vectors.items():
    if v[5] > 0.6:  # high fanout
        print(f"  {node:<25} fanout={v[5]:.3f}  criticality={v[4]:.3f}")

---
## Example 2: ETL / Data Pipeline

A data pipeline with ingestion, transformation, and loading stages. Common structural risk: a shared transformation step that every downstream report depends on.

In [ ]:
# Data pipeline
etl_pipeline = [
    # Ingestion sources
    ("salesforce-sync",   "raw-landing"),
    ("stripe-webhook",    "raw-landing"),
    ("postgres-cdc",      "raw-landing"),
    ("s3-file-drop",      "raw-landing"),
    
    # Transformation
    ("raw-landing",       "dedup-transform"),
    ("dedup-transform",   "normalize"),
    ("normalize",         "enrich-geo"),
    ("normalize",         "enrich-company"),
    
    # Loading
    ("enrich-geo",        "warehouse-load"),
    ("enrich-company",    "warehouse-load"),
    ("warehouse-load",    "dashboard-refresh"),
    ("warehouse-load",    "ml-feature-store"),
    ("warehouse-load",    "export-to-s3"),
]

result_etl = encode(etl_pipeline)
print("ETL PIPELINE STRUCTURAL ANALYSIS")
print("=" * 50)
print(result_etl.table)
print()
print(report(result_etl))

---
## Example 3: Drift â€” What Changed Between Releases?

Someone adds a new data source and a direct path to the ML feature store. Did this change the structural risk profile?

In [ ]:
# New version: added kafka stream and direct ML path
etl_v2 = etl_pipeline + [
    ("kafka-stream",      "raw-landing"),
    ("normalize",         "ml-feature-store"),  # bypass warehouse
]

result_v2 = encode(etl_v2)

changes = drift(result_etl, result_v2)
print("STRUCTURAL DRIFT: ETL v1 -> v2")
print("=" * 50)
print(f"Added: kafka-stream -> raw-landing")
print(f"Added: normalize -> ml-feature-store (bypass warehouse)")
print()

for node, deltas in sorted(changes.items()):
    print(f"{node}:")
    for dim, delta in deltas.items():
        direction = "+" if delta > 0 else ""
        print(f"  {dim}: {direction}{delta:.4f}")
    print()

print("RISKS AFTER CHANGE:")
print(report(result_v2))

---
## Your Own Pipeline

Replace the edges below with your CI/CD or data pipeline.

In [ ]:
my_pipeline = [
    ("step-a", "step-b"),
    ("step-a", "step-c"),
    ("step-b", "step-d"),
    ("step-c", "step-d"),
]

my_result = encode(my_pipeline)
print(my_result.table)
print()
print(report(my_result))

---

**Next:** [05 - AI Agent Pipelines](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/05_ai_agent_pipelines.ipynb)

*Patent pending. Application #63/994,075.*

## Parse a real workflow tree

If your CI lives in `.github/workflows/`, `from_github_actions` gives you the
job-dependency graph for free:

```python
import semanticembed as se
edges = se.extract.from_github_actions(".github/workflows")
result = se.encode(edges)
print(result.table)
```

For a one-shot scan of any repo:

```python
edges, sources = se.extract.from_directory(".")
# sources includes "github-actions" if workflow files were found
```
